# Brain Tumor Detection - Exploration & Prototyping

This notebook mirrors the training pipeline used in `train.py`. It's meant for exploring the dataset and experimenting with the model before running the full script. For the actual production training run, use `python train.py` from the project root.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


## 1. Dataset paths

The notebook lives in `notebooks/`, so paths are relative to the project root (one level up).

In [ ]:
TRAIN_DIR = "../dataset/Training"
TEST_DIR = "../dataset/Testing"
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

print(os.listdir(TRAIN_DIR))


## 2. Data generators

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2,
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", subset="training", shuffle=True, seed=42,
)

validation_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", subset="validation", shuffle=False, seed=42,
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False,
)

CLASS_NAMES = list(train_generator.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)
print(train_generator.class_indices)


## 3. Peek at a few training images

In [ ]:
images, labels = next(train_generator)

plt.figure(figsize=(10, 6))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    # undo MobileNetV2 preprocessing just for display
    img = (images[i] + 1) / 2
    plt.imshow(img)
    plt.title(CLASS_NAMES[np.argmax(labels[i])])
    plt.axis("off")
plt.tight_layout()
plt.show()


## 4. Build the model (MobileNetV2 transfer learning)

In [ ]:
base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
predictions = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(learning_rate=0.0001), loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()


## 5. Quick training run

Just a couple of epochs here to sanity-check the pipeline. The full run with callbacks and evaluation lives in `train.py`.

In [ ]:
history = model.fit(train_generator, validation_data=validation_generator, epochs=2)


## 6. Accuracy / loss curves

In [ ]:
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Val Accuracy")
plt.legend()
plt.title("Accuracy")
plt.show()


## 7. Try a single prediction

In [ ]:
from tensorflow.keras.preprocessing import image

sample_path = os.path.join(TEST_DIR, CLASS_NAMES[0])
sample_file = os.listdir(sample_path)[0]

img = image.load_img(os.path.join(sample_path, sample_file), target_size=IMAGE_SIZE)
img_array = image.img_to_array(img)
img_array = preprocess_input(img_array)
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)
predicted_class = CLASS_NAMES[np.argmax(prediction)]
confidence = np.max(prediction) * 100

print(f"Predicted: {predicted_class} ({confidence:.2f}% confidence)")


---
Once you're happy with the approach, run the full training pipeline with:

```
python train.py
```

This saves the model to `models/brain_tumor_model.keras`, which `app.py` and `predict.py` use for inference.